# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Keywords: {metadata['keywords']}")
print(f"Data collection timeframe: {metadata['dataCollectionTimeframe']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` for reproducibility and clarity per Croissant schema guidelines.

In [ ]:
# List all record sets and their @ids
record_sets = dataset.record_sets()
print('Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For the first record set, list all fields and columns
if record_sets:
    first_rs_id = record_sets[0]['@id']
    rs_obj = dataset.record_set(first_rs_id)
    print(f"\nFields in record set @id={first_rs_id}:")
    for field in rs_obj.fields():
        print(f"  - Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")

    print(f"\nColumns in record set @id={first_rs_id}:")
    for col in rs_obj.columns():
        print(f"  - Column @id: {col['@id']}, name: {col.get('name', 'N/A')}")
else:
    print('No record sets found in this dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare record sets for extraction by @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded DataFrame for record set @id={rs_id}: Columns={df.columns.tolist()}")
        print(df.head(2))
    except Exception as exc:
        print(f"Record set @id={rs_id} unable to load records: {exc}")

# Select one record set for further analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nUsing record set @id={main_record_set_id} for analysis.")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations include removing outliers, transforming distributions, or grouping data by attributes using their `@id`s.

In [ ]:
import numpy as np

# Example fields for EDA (replace with actual @id from previous listing)
# Suppose 'gad7_score', 'phq9_score', and 'village' are fields in the record set
# Example IDs (replace as needed with actual from the dataset)
numeric_field_id = 'gad7_score'  # Use the @id or column name as seen in dataframe
group_field_id = 'village'       # Use the @id or column name as seen in dataframe

df = dataframes[main_record_set_id]

# Filter records by threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    if std > 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print(f"Standard deviation of {numeric_field_id} is zero; normalization not applied.")

    # Group and aggregate
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll plot the distribution of GAD-7 scores and mean scores by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of GAD-7 scores
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Plot mean score by village (if grouped_df exists)
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 6))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- **Data loaded** from Croissant schema URL with `mlcroissant`.
- **Record set and fields** explored and referenced using their `@id`s.
- **EDA** applied to GAD-7 scores: records filtered by threshold, normalized, and grouped by village.
- **Visualizations** highlighted the distribution and differences across villages.
- **Next steps**: use field `@id`s for more detailed analysis of additional indicators (e.g., PHQ-9, PSQ), demographic data, or temporal trends.

> This approach ensures reproducible, FAIR-compliant analysis pipelines for mental health survey datasets in Africa.